# সহজ ভাষায় Notebook Guide

এই notebook-এ theory এবং code পাশাপাশি শেখানো হয়েছে। Technical term English-এ থাকবে, আর explanation Bangla-তে—যাতে code-এর language-এর সাথে পরিচিত থেকেও concept সহজে বোঝা যায়।

## কীভাবে ব্যবহার করবেন?

1. Cell উপর থেকে নিচে sequence অনুযায়ী run করুন।
2. Run করার আগে expected output কী হতে পারে লিখে ভাবুন।
3. Output-এর metric, shape এবং visualization explanation-এর সাথে compare করুন।
4. Error হলে import, file path, data shape এবং dependency একে একে check করুন।
5. Notebook শেষে নিজের ভাষায় লিখুন: এটি কোন problem solve করেছে, কীভাবে করেছে এবং limitation কী।

> **Important:** Notebook-এর সব cell successful run হলেই result correct প্রমাণ হয় না; data leakage, wrong assumption এবং misleading metric-ও validate করতে হবে।

# Structured Outputs with LangChain: Resume-to-JSON Converter

> **Focus Area:** স্ট্রাকচার্ড আউটপুট ও ফ্রেমওয়ার্ক (Structured Outputs & Frameworks)
> **Topics:** Getting JSON from LLMs; LangChain; Prompt Templates; Chains of Reasoning
> **Achievement:** একটা CV/সিভি টেক্সট পড়ে নাম, স্কিল, কাজের অভিজ্ঞতা এবং এডুকেশন — সবকিছু একটা Pydantic-ভ্যালিডেটেড JSON স্কিমায় বের করে আনা, লোকাল LLM (Ollama + `llama3.1:8b`) ব্যবহার করে।

---

> **Setup (একবারই করতে হবে):**
> ```bash
> ollama pull llama3.1:8b
> pip install langchain langchain-ollama pydantic
> ```
> Ollama সার্ভার লোকালি চলমান থাকতে হবে (`ollama serve`) — এই নোটবুকের কোড সেল এক্সিকিউট করা হয়নি (এই এনভায়রনমেন্টে Ollama নেই), কিন্তু নিচের প্রতিটা কোড সেল সরাসরি রান-এবল, শুধু লোকাল Ollama + `llama3.1:8b` লাগবে।

---

## 1. Topic: Getting JSON from LLMs, LangChain, Prompt Templates, Chains of Reasoning

বেশিরভাগ প্রোডাকশন সিস্টেমে LLM-এর আউটপুট মানুষ পড়ে না — আরেকটা প্রোগ্রাম সেটা পড়ে, ডেটাবেসে সেভ করে। তার জন্য LLM-এর আউটপুট **টেক্সট** না হয়ে **স্ট্রাকচার্ড ডেটা** (JSON) হতে হয়। এই নোটবুকে আমরা হাতে-কলমে দেখব:

* **Getting JSON from LLMs** — একটা কড়া প্রম্পট (স্কিমা-সহ) দিয়ে LLM-কে নির্দিষ্ট ফরম্যাটে আউটপুট দিতে বাধ্য করা।
* **LangChain** — `PromptTemplate`, `ChatOllama`, এবং `PydanticOutputParser`-কে একটা `|` (pipe) চেইনে জোড়া দেওয়া।
* **Prompt Templates** — রিইউজেবল প্রম্পট গঠন।
* **Chains of Reasoning** — একটা জটিল extraction টাস্ককে একাধিক ধাপে ভাগ করে চেইন করা।

**Achievement:** একটা **Resume-to-JSON** কনভার্টার — CV টেক্সট থেকে অভিজ্ঞতা, স্কিল, এডুকেশন বের করে একটা পরিষ্কার, ডেটাবেসে-সেভ-করার-উপযোগী JSON ফরম্যাটে দেওয়া।

---

## 2. Why It Is Related

একটা ATS (Applicant Tracking System) হাজার হাজার সিভি প্রসেস করার সময়, সেই ডেটা একটা ডেটাবেসে সার্চেবল ফরম্যাটে থাকতে হয় — "৫ বছর Python অভিজ্ঞতা আছে এমন সব ক্যান্ডিডেট দেখাও" — এই ধরনের কোয়েরি চালানো সম্ভব হয় শুধু তখনই, যখন ডেটা structured থাকে। LLM দিয়ে unstructured টেক্সট (সিভির ফ্রি-ফর্ম লেখা) থেকে structured ডেটা বের করা — এই প্যাটার্নকে বলে **Information Extraction**, এবং এটা ইনভয়েস প্রসেসিং, মেডিকেল রেকর্ড পার্সিং, কন্ট্র্যাক্ট অ্যানালাইসিস — এরকম অসংখ্য রিয়েল-ওয়ার্ল্ড এআই অ্যাপ্লিকেশনের ভিত্তি। Week 10-এ আমরা CV থেকে "অর্থ" বের করেছিলাম (semantic matching, embeddings দিয়ে); এখন আমরা CV থেকে "ডেটা" বের করছি (structured extraction) — দুটোই ভিন্ন কিন্তু পরিপূরক স্কিল।

---

## 3. How It Works

### 3.1 তিনটা পদ্ধতিতে JSON বের করা যায়

```
পদ্ধতি ১: Prompt Engineering       → সহজ, কিন্তু মডেল মাঝেমধ্যে ফরম্যাট ভুল করে
পদ্ধতি ২: Structured Output Mode   → প্রোভাইডার নিজেই আউটপুট ভ্যালিডেট করে
পদ্ধতি ৩: Output Parser + Retry    → parse করার চেষ্টা, ব্যর্থ হলে এরর-সহ আবার জিজ্ঞেস
```

লোকাল LLM (Ollama, `llama3.1:8b`) দিয়ে আমরা পদ্ধতি ১ আর ৩ মিলিয়ে ব্যবহার করব — একটা কড়া প্রম্পট (স্কিমা-সহ) দিয়ে শুরু, তারপর আউটপুট parse করে ব্যর্থ হলে retry।

### 3.2 LangChain-এর পাইপলাইন

```
PromptTemplate ──▶ ChatOllama (llama3.1:8b) ──▶ PydanticOutputParser ──▶ ResumeData object
      │                      │                          │
   ভ্যারিয়েবল বসানো        মডেল কল                 JSON স্ট্রিং থেকে
                                                    validated Pydantic-এ রূপান্তর
```

LangChain-এর `Runnable`/`|` (pipe) সিনট্যাক্স দিয়ে এই পুরো পাইপলাইনকে একটা লাইনে চেইন করা যায়: `chain = prompt | llm | parser`।

### 3.3 Chains of Reasoning — জটিল কাজ ধাপে ভাগ করা

```
Step 1: Extract raw sections (Experience, Skills, Education) — একটা LLM কল
Step 2: প্রতিটা raw সেকশনকে structured JSON ফিল্ডে রূপান্তর — আরেকটা LLM কল
Step 3: Validate & merge — কোড দিয়ে (Pydantic parser), LLM দিয়ে না
```

একটা ধাপের আউটপুট পরের ধাপের ইনপুট হয় — এই নোটবুকের শেষ সেকশনে আমরা এই ২-ধাপের চেইনটা বানাব।

---

## 4. Achievement: Resume-to-JSON Converter — Code Walkthrough

নিচের কোড সেলগুলো ধাপে ধাপে বানাবে: (৪.১) Pydantic স্কিমা, (৪.২) Prompt Template + টয় CV, (৪.৩) extraction chain, (৪.৪) retry লজিক, (৪.৫) ২-ধাপের chain-of-reasoning ডেমো।

> **নোট:** নিচের সব কোড সেল সরাসরি রান-এবল যদি লোকালি Ollama চলমান থাকে এবং `llama3.1:8b` pull করা থাকে — কিন্তু এই এনভায়রনমেন্টে Ollama সার্ভার নেই বলে সেলগুলো এক্সিকিউট করা হয়নি (`execution_count: null`, কোনো output নেই)।

---

In [ ]:
# Imports — LangChain + Ollama + Pydantic
from typing import List, Optional

from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.exceptions import OutputParserException
from langchain_core.runnables import RunnableLambda
from langchain_ollama import ChatOllama

# লোকালি চলা Ollama সার্ভারের llama3.1:8b মডেলের সাথে কানেক্ট করা হচ্ছে
# temperature=0 → extraction টাস্কে ডিটারমিনিস্টিক আউটপুট চাই, ক্রিয়েটিভিটি না
llm = ChatOllama(model="llama3.1:8b", temperature=0)


### 4.1 Pydantic Schema — কেন স্কিমা ভ্যালিডেশন জরুরি

শুধু "JSON পার্স হলো কিনা" যথেষ্ট না — আউটপুটে সঠিক ফিল্ড আছে কিনা, সঠিক টাইপ আছে কিনা (যেমন কোনো সংখ্যার ফিল্ড আসলেই সংখ্যা, স্ট্রিং না) — এটাও যাচাই করা জরুরি। **Pydantic** মডেল ব্যবহার করে এই স্কিমা ভ্যালিডেশন স্বয়ংক্রিয় হয়ে যায় — ভুল ফরম্যাট এলে সাথে সাথে ধরা পড়ে, সাইলেন্টলি ভুল ডেটা ডেটাবেসে ঢুকে যাওয়ার আগেই।

নিচে `WorkExperience` আর `Education`-কে আলাদা nested model হিসেবে রাখা হয়েছে (একটা সিঙ্গেল `years_experience` ফিল্ডের বদলে) — কারণ একজন ক্যান্ডিডেটের একাধিক জব থাকতে পারে, প্রতিটার duration আলাদা; সব জব মিলিয়ে একটা সংখ্যায় গুঁজে দিলে "কোন কোম্পানিতে কত বছর" তথ্যটাই হারিয়ে যায়।

In [ ]:
# Pydantic স্কিমা — প্রতিটা ফিল্ডের description parser-কে (এবং মডেলকে) বলে দেয় কী আশা করা হচ্ছে
class WorkExperience(BaseModel):
    """একটা একক জব এন্ট্রি — company/title/years, প্রতিটা জব আলাদা এন্ট্রি হিসেবে রাখা হচ্ছে"""

    company: str = Field(description="কোম্পানির নাম")
    title: str = Field(description="জব টাইটেল/পজিশন")
    years: float = Field(description="ঐ পজিশনে কত বছর কাজ করেছেন, সংখ্যা হিসেবে (স্ট্রিং না)")


class Education(BaseModel):
    """একটা এডুকেশন এন্ট্রি"""

    institution: str = Field(description="প্রতিষ্ঠানের নাম")
    degree: str = Field(description="ডিগ্রি বা সার্টিফিকেটের নাম")
    year: Optional[int] = Field(default=None, description="গ্র্যাজুয়েশনের সাল, উল্লেখ না থাকলে null")


class ResumeData(BaseModel):
    """সম্পূর্ণ CV থেকে বের করা structured ডেটা — ডেটাবেসে সেভ হওয়ার আগের শেষ ভ্যালিডেশন চেকপয়েন্ট"""

    name: str = Field(description="ক্যান্ডিডেটের পূর্ণ নাম")
    email: Optional[str] = Field(default=None, description="ইমেইল অ্যাড্রেস, উল্লেখ না থাকলে null")
    skills: List[str] = Field(default_factory=list, description="টেকনিক্যাল স্কিলের লিস্ট")
    experience: List[WorkExperience] = Field(default_factory=list, description="কাজের অভিজ্ঞতা, প্রতিটা জব আলাদা এন্ট্রি")
    education: List[Education] = Field(default_factory=list, description="এডুকেশন হিস্ট্রি")


### 4.2 Prompt Template + টয় CV

`PromptTemplate` ব্যবহার করলে একই প্রম্পট স্ট্রাকচার হাজারটা ভিন্ন CV-তে reuse করা যায়, কোড রিপিট না করে — ঠিক যেমন একটা ফাংশন একবার লিখে বারবার কল করা হয়। `PydanticOutputParser.get_format_instructions()` স্বয়ংক্রিয়ভাবে স্কিমার বর্ণনা থেকে একটা ফরম্যাট-ইনস্ট্রাকশন টেক্সট বানিয়ে দেয়, যেটা প্রম্পটে বসিয়ে দেওয়া হয় — মডেলকে ঠিক কী ফরম্যাটে উত্তর দিতে হবে তা বলে দেওয়ার জন্য।

In [ ]:
# টয় সিভি — সিন্থেটিক, ৩-৪ বাক্যের একটা প্যারাগ্রাফ
toy_resume_text = (
    "Jamil Ahmed is a software engineer reachable at jamil.ahmed@example.com. "
    "He has 4 years of experience as a Backend Developer at Pathao and 2 years as a Junior "
    "Developer at Brain Station 23, working mainly with Python, Django, and PostgreSQL. "
    "He completed his BSc in Computer Science and Engineering from BUET in 2019."
)

# parser.get_format_instructions() স্কিমা থেকে স্বয়ংক্রিয়ভাবে JSON ফরম্যাট-নির্দেশনা তৈরি করে দেয়
parser = PydanticOutputParser(pydantic_object=ResumeData)

extraction_prompt = PromptTemplate(
    template=(
        "তুমি একজন নির্ভুল ডেটা এক্সট্রাকশন অ্যাসিস্ট্যান্ট। নিচের সিভি টেক্সট থেকে "
        "নাম, ইমেইল, স্কিল, কাজের অভিজ্ঞতা এবং এডুকেশন বের করে দাও।\n\n"
        "{format_instructions}\n\n"
        "সিভি টেক্সট:\n{resume_text}\n"
    ),
    input_variables=["resume_text"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)


### 4.3 Extraction Chain — `prompt | llm | parser`

LangChain-এর `|` (pipe) সিনট্যাক্স দিয়ে তিনটা ধাপ (prompt বসানো → মডেল কল → parse করা) একটা `Runnable` চেইনে জোড়া লাগানো হয়। `chain.invoke(...)` কল করলে পুরো পাইপলাইন একবারে রান হয় এবং শেষে একটা validated `ResumeData` অবজেক্ট ফেরত আসে, raw টেক্সট বা dict না।

In [ ]:
# prompt | llm | parser — তিনটা ধাপ একটা Runnable চেইনে জোড়া লাগানো হলো
extraction_chain = extraction_prompt | llm | parser

# chain.invoke() কল করলে prompt ফরম্যাট হয় → llm কল হয় → আউটপুট parse হয়ে ResumeData ফেরত আসে
# (এই এনভায়রনমেন্টে Ollama সার্ভার না থাকায় এই লাইনটা রান করা হয়নি)
result = extraction_chain.invoke({"resume_text": toy_resume_text})

print(type(result))          # <class '__main__.ResumeData'>
print(result.name)           # "Jamil Ahmed"
print(result.skills)         # ["Python", "Django", "PostgreSQL", ...]
print(result.model_dump_json(indent=2))


### 4.4 Retry Strategy — মডেল ভুল করলে কী করবেন

লোকাল LLM (৮B প্যারামিটার) মাঝেমধ্যে অসম্পূর্ণ বা ভুল-ফরম্যাটেড JSON দিতে পারে, বড় ক্লাউড মডেলের তুলনায় বেশি হতে পারে। একটা robust পাইপলাইনে থাকে: (১) parse করার চেষ্টা, (২) ব্যর্থ হলে "তোমার আগের আউটপুট ভ্যালিড ছিল না" — এই এরর ফিডব্যাকসহ আবার কল করা, (৩) নির্দিষ্ট সংখ্যক (এখানে ৩বার) চেষ্টার পরও ব্যর্থ হলে exception রেইজ করে জানানো — silently ভুল ডেটা ফেরত না দিয়ে।

In [ ]:
def extract_with_retry(resume_text: str, max_attempts: int = 3) -> ResumeData:
    """
    পদ্ধতি ৩: parse করার চেষ্টা, ব্যর্থ হলে এরর মেসেজসহ মডেলকে আবার জিজ্ঞেস করা।
    max_attempts বার চেষ্টার পরও ব্যর্থ হলে RuntimeError রেইজ করে — silently ভুল ডেটা না দিয়ে।
    """
    error_feedback = ""

    for attempt in range(1, max_attempts + 1):
        prompt_text = extraction_prompt.format(resume_text=resume_text)

        if error_feedback:
            # আগের এরর মেসেজটা প্রম্পটে যোগ করে দেওয়া হচ্ছে, যাতে মডেল নিজের ভুল দেখে ঠিক করতে পারে
            prompt_text += (
                f"\n\nতোমার আগের আউটপুট ভ্যালিড ফরম্যাটে ছিল না। এরর মেসেজ:\n{error_feedback}\n"
                "আবার চেষ্টা করো, শুধু ভ্যালিড JSON দাও, অতিরিক্ত কোনো টেক্সট ছাড়া।"
            )

        raw_output = llm.invoke(prompt_text)

        try:
            return parser.parse(raw_output.content)
        except OutputParserException as exc:
            error_feedback = str(exc)
            print(f"Attempt {attempt} failed, retrying with error feedback...")

    raise RuntimeError(
        f"{max_attempts} বার চেষ্টার পরও ভ্যালিড JSON পাওয়া যায়নি — ম্যানুয়াল রিভিউ প্রয়োজন।"
    )


# result = extract_with_retry(toy_resume_text)


### 4.5 Chain-of-Reasoning Demo — দুই ধাপে ভাগ করে extraction

একটা জটিল extraction টাস্ক একটা LLM কলে ভালোভাবে না হয়ে দুই ধাপে ভাগ করে করা যায়:

```
Step 1: Extract raw sections (Experience, Skills, Education) — separate LLM call
Step 2: Structure each section into JSON fields — separate LLM call
Step 3: Validate & merge — code (Pydantic parser), LLM না
```

Step 1-এর আউটপুট Step 2-এর ইনপুট হয়ে যায় — LangChain-এর `|` সিনট্যাক্স দিয়ে এই চেইনটা পরিষ্কারভাবে লেখা যায়। মাঝখানে `RunnableLambda` দিয়ে Step 1-এর `AIMessage` আউটপুটকে Step 2-এর প্রম্পটের জন্য dict-এ রূপান্তর করা হচ্ছে।

In [ ]:
# Step 1: raw সেকশন বের করা (Experience, Skills, Education আলাদা টেক্সট ব্লক হিসেবে)
section_extraction_prompt = PromptTemplate.from_template(
    "নিচের সিভি টেক্সট থেকে তিনটা সেকশন আলাদা করে লেখো — Experience, Skills, Education। "
    "প্রতিটা সেকশনের হেডিং এবং তার নিচে সংশ্লিষ্ট তথ্য দাও, অতিরিক্ত কিছু লিখো না।\n\n"
    "সিভি টেক্সট:\n{resume_text}"
)

# Step 2: raw সেকশন-টেক্সট থেকে ফাইনাল Pydantic স্কিমায় স্ট্রাকচার করা
structuring_prompt = PromptTemplate(
    template=(
        "নিচের সেকশন-ভাগ-করা টেক্সট থেকে একটা structured resume বানাও।\n\n"
        "{format_instructions}\n\n"
        "সেকশন টেক্সট:\n{sections_text}\n"
    ),
    input_variables=["sections_text"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# Step 1-এর AIMessage আউটপুটকে Step 2-এর প্রম্পট-ইনপুট dict-এ রূপান্তর করছে
to_sections_input = RunnableLambda(lambda msg: {"sections_text": msg.content})

# দুই ধাপ জোড়া লাগিয়ে একটা Runnable চেইন — Step 3 (validate & merge) parser নিজেই করে, কোড দিয়ে
two_step_chain = (
    section_extraction_prompt
    | llm
    | to_sections_input
    | structuring_prompt
    | llm
    | parser
)

# result = two_step_chain.invoke({"resume_text": toy_resume_text})


## 5. Summary

আজকে আমরা structured extraction পাইপলাইনের মূল বিল্ডিং ব্লকগুলো হাতে-কলমে বানালাম:

1. **Pydantic Schema** (`ResumeData`, `WorkExperience`, `Education`) — আউটপুটের ফিল্ড আর টাইপ ভ্যালিডেট করার জন্য।
2. **Prompt Template** (`extraction_prompt`) — `get_format_instructions()` দিয়ে স্কিমা থেকে স্বয়ংক্রিয় ফরম্যাট-নির্দেশনা তৈরি, একই টেমপ্লেট যেকোনো CV-তে reuse-এবল।
3. **Extraction Chain** (`prompt | llm | parser`) — LangChain-এর `|` সিনট্যাক্স দিয়ে prompt-formatting, মডেল-কল, এবং parsing একসাথে জোড়া।
4. **Retry Strategy** (`extract_with_retry`) — parse ব্যর্থ হলে এরর ফিডব্যাকসহ পুনরায় চেষ্টা, নির্দিষ্ট সীমার পর স্পষ্ট এরর রেইজ।
5. **Chain-of-Reasoning** (`two_step_chain`) — জটিল extraction টাস্ককে দুই ধাপে ভাগ করে (raw sections → structured JSON), `RunnableLambda` দিয়ে ধাপগুলো ব্রিজ করা।

---

## 🧠 Brain Teasers & Exercises (নিজে চেষ্টা করুন)

1. **Schema Design:** উপরের `ResumeData` স্কিমায় আরও কী ফিল্ড যোগ করলে ভালো হতো মনে করেন — যেমন `certifications` বা `total_years_experience`? `WorkExperience`-এ `years` কেন `float` করা হলো, `int` না কেন — চিন্তা করুন কোন কেসে এটা গুরুত্বপূর্ণ হতে পারে।
2. **Ambiguous CV Test:** `toy_resume_text`-এর মতো এমন একটা CV স্নিপেট লিখুন যেখানে "স্কিল" আর "অভিজ্ঞতা" আলাদা করা কঠিন (যেমন "৩ বছর ধরে Python ব্যবহার করে ডেটা পাইপলাইন বানিয়েছি")। এটা `extraction_chain.invoke()`-এ পাস করলে `skills` আর `experience` ফিল্ড দুটোয় কী আসবে বলে মনে হয় — নিজে লোকাল Ollama দিয়ে টেস্ট করে দেখুন।
3. **Retry Limit:** `extract_with_retry`-তে `max_attempts=3`-এর পরও ভ্যালিড JSON না এলে বর্তমানে `RuntimeError` রেইজ হয়। প্রোডাকশন সিস্টেমে তখন কী করা উচিত — ইউজারকে এরর দেখানো, নাকি আংশিক ডেটা (যা এতদূর parse হয়েছে) সেভ করে "ম্যানুয়াল রিভিউ প্রয়োজন" ফ্ল্যাগ করা? `extract_with_retry` ফাংশনটা এডিট করে দ্বিতীয় approach-টা implement করার চেষ্টা করুন।